# 06 · DPO: preference tuning from scratch

Learn from *comparisons* instead of gold answers: build preference data, fit a Bradley–Terry reward model, then write
the DPO loss yourself and train a tiny policy against a frozen reference — watching the log-probabilities, the
implicit reward margin and the effect of β.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min, no downloads · **Needs:**
[02 · SFT with loss masking](02_sft_from_scratch_with_loss_masking.ipynb) (sequence log-probs, masking)

## Why this matters in interviews

- "What is DPO and why did it largely replace RLHF?" is asked in nearly every LLM training interview. The strong
  answer writes the loss, names the frozen reference model and explains β.
- Reward models and their failure modes (length bias, reward hacking, over-optimisation) are the follow-up.
- Knowing the alternatives by one line each — IPO, KTO, ORPO, SimPO, GRPO — and *when* each fits the data you have
  is what separates practitioners from readers.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tn07` | What is DPO and why did it largely replace RLHF? |
| `tn08` | What is RLHF, in detail? |
| `tn17` | What is a reward model, and what goes wrong with it? |
| `tn37` | What is the difference between SFT and preference tuning, in terms of what each can teach? |
| `tn19` | What are ORPO and KTO, and why would you use them over DPO? |
| `tn09` | What is GRPO and what problem does it solve? |
| `tn16` | What is constitutional AI / RLAIF? |
| `tn38` | What is the "alignment tax" and does it still exist? |

In [ ]:
import copy, math, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_num_threads(1)
torch.manual_seed(0); np.random.seed(0)
rng = np.random.default_rng(0)

## 1. Preference data

**In plain English:** for SFT you must *write* the ideal answer. For preference tuning you only have to say which
of two answers is *better* — much faster for a human, and it captures things that are easy to recognise but hard to
write down: tone, concision, hedging, a good refusal.

One record is `(prompt, chosen, rejected)`:

In [ ]:
pairs_text = [
    {"prompt": "How do I reset my password?",
     "chosen": "Go to Settings > Security > Reset password; the link arrives within a minute.",
     "rejected": "Just figure it out, it's in the settings somewhere."},
    {"prompt": "Can you summarise this contract clause?",
     "chosen": "It lets either party end the deal with 30 days' written notice.",
     "rejected": "Contracts are complicated. You should read it yourself."},
]
for p in pairs_text:
    print(f"prompt  : {p['prompt']}\n  chosen  : {p['chosen']}\n  rejected: {p['rejected']}")

Where pairs come from: humans comparing two samples from the SFT model (the classic RLHF recipe), an LLM judge
applying a written rubric (RLAIF / constitutional AI, `tn16`), or product signals (the regenerated answer vs the one
the user kept). Thumbs up/down on *single* answers are **not** pairs — that is KTO's data (section 6).

## 2. Reward models and Bradley–Terry

A reward model maps (prompt, response) to one number `r`. The **Bradley–Terry** model turns two rewards into a
preference probability:

```
P(chosen ≻ rejected) = σ(r_chosen − r_rejected)          σ(z) = 1 / (1 + e^(−z))
loss = −log σ(r_chosen − r_rejected)
```

Only the **difference** matters: a gap of 0 is a coin flip, a gap of 2 is 88%, a gap of 4 is 98%.

In [ ]:
gap = np.linspace(-6, 6, 200)
plt.figure(figsize=(7, 3.2))
plt.plot(gap, 1 / (1 + np.exp(-gap)))
for g_ in (0, 2, 4):
    plt.plot(g_, 1 / (1 + np.exp(-g_)), "o", color="#e8590c")
    plt.annotate(f"gap {g_}: {1 / (1 + np.exp(-g_)):.0%}", (g_, 1 / (1 + np.exp(-g_))), xytext=(g_ - 3.2, 1 / (1 + np.exp(-g_)) + 0.05))
plt.xlabel("r_chosen − r_rejected"); plt.ylabel("P(chosen ≻ rejected)"); plt.title("Bradley–Terry: preference = sigmoid of the reward gap")
plt.show()

### Fit a reward model — and watch it learn a bias

Simulate labellers. Each response has three features: `helpful` (what we *want* rewarded), `polite`, and `length`.
The labellers' true taste is mostly helpfulness and politeness — but, like real annotators, they also slightly
prefer **longer** answers. A linear reward model `r = w · features` trained with the Bradley–Terry loss recovers
their taste, bias included.

In [ ]:
true_w = np.array([2.0, 1.0, 0.6])                # helpful, polite, length (the labellers' hidden taste)
names = ["helpful", "polite", "length"]
n = 2000
feat_a, feat_b = rng.normal(size=(n, 3)), rng.normal(size=(n, 3))
p_a = 1 / (1 + np.exp(-(feat_a - feat_b) @ true_w))
a_wins = rng.random(n) < p_a
chosen = np.where(a_wins[:, None], feat_a, feat_b)
rejected = np.where(a_wins[:, None], feat_b, feat_a)

w = torch.zeros(3, requires_grad=True)
opt = torch.optim.Adam([w], lr=0.05)
C, R = torch.tensor(chosen, dtype=torch.float32), torch.tensor(rejected, dtype=torch.float32)
for step in range(300):
    loss = -F.logsigmoid(C @ w - R @ w).mean()                 # Bradley-Terry loss
    opt.zero_grad(); loss.backward(); opt.step()
acc = ((C @ w - R @ w) > 0).float().mean().item()
for nm, t, l in zip(names, true_w, w.detach().numpy()):
    print(f"{nm:8s} true weight {t:.2f}   learned {l:.2f}")
print(f"reward model agrees with the labels on {acc:.1%} of pairs (the labels themselves are noisy)")
assert np.allclose(w.detach().numpy(), true_w, atol=0.3)

The reward model faithfully learned the **length** preference too. Now optimise a policy hard against it: the cheapest
way to raise the reward is to write longer — without being more helpful. That is how RLHF models became verbose,
and it is the simplest case of **reward hacking** (`tn17`).

In [ ]:
lengths = np.linspace(0, 4, 50)
learned = w.detach().numpy()
plt.figure(figsize=(7, 3.2))
plt.plot(lengths, learned[2] * lengths, label="reward gained by padding the answer (helpful, polite fixed)")
plt.xlabel("extra length (std units)"); plt.ylabel("reward-model score gain")
plt.title("A reward model with a length bias pays for verbosity"); plt.legend(); plt.show()

Mitigations worth naming: a KL penalty toward the reference model (the leash in PPO — below), length-normalised or
length-controlled rewards, ensembles of reward models, and refreshing the reward model on outputs of the current
policy (it was trained on the SFT model's outputs and drifts out of distribution).

## 3. RLHF (PPO) vs DPO

```
RLHF:  SFT model ──► sample pairs ──► humans rank ──► train reward model ──► PPO: policy maximises reward − β·KL(π‖π_ref)
DPO:   SFT model ──► sample pairs ──► humans rank ─────────────────────────► DPO loss directly on (prompt, chosen, rejected)
```

| | RLHF with PPO | DPO |
|---|---|---|
| models in memory | policy, reference, reward model, value/critic | policy, reference (frozen) |
| training loop | on-line: generate rollouts, score, PPO update | off-line: a classification-style loss on fixed pairs |
| stability / tuning | notoriously finicky (KL coef, clipping, value loss, rollout size) | a few hyperparameters (β, LR); stable |
| cost | high (generation inside the loop) | about the cost of SFT on pairs |
| exploration | can explore beyond the data, against a live reward | limited to the pairs you collected |
| best when | reward is a program or a live signal; online improvement | you have (or can make) preference pairs — the common case |

DPO's insight: the KL-regularised RLHF objective has a closed-form optimal policy, `π*(y|x) ∝ π_ref(y|x)·exp(r(x,y)/β)`.
Solve that for `r` and substitute it into the Bradley–Terry loss: the reward model disappears, leaving a loss on the
policy's own log-probabilities. The **implicit reward** of a response is `β · (log π(y|x) − log π_ref(y|x))`.

```
L_DPO = −log σ( β · [ (log π(y_c|x) − log π_ref(y_c|x)) − (log π(y_r|x) − log π_ref(y_r|x)) ] )
```

## 4. DPO from scratch on a tiny policy

**The toy.** Six support prompts. For each, three answers written in a 20-word vocabulary: *helpful*, *curt* and a
*rambling* one. The policy is a tiny GRU language model conditioned on the prompt. Step 1 is SFT on **all three**
styles — like a real SFT model whose answers vary in quality. That SFT model becomes the frozen **reference**. Step 2
is DPO on pairs that only ever compare helpful against curt; the rambling answer never appears in a pair
with the helpful answer as `chosen`.

In [ ]:
vocab = ["<pad>", "<eos>", "sure", "here", "is", "how", "go", "to", "settings", "then", "click", "reset",
         "no", "idea", "later", "busy", "ask", "someone", "else", "sorry"]
stoi = {w_: i for i, w_ in enumerate(vocab)}
PAD, EOS = 0, 1
helpful = ["sure here is how", "go to settings then click reset", "sure go to settings",
           "sure here is how then click reset", "go to settings then click", "sure click reset"]
curt = ["no idea", "ask someone else", "busy ask later", "no idea ask someone else", "sorry busy", "later"]
rambling = ["sorry sorry here is is how how", "go go to to settings settings", "then then click click",
            "here here is is how how", "sorry then then click", "is is how how then"]
P = len(helpful)
enc = lambda s: [stoi[w_] for w_ in s.split()] + [EOS]

def to_batch(texts):
    T = max(len(enc(t)) for t in texts)
    ids = torch.full((len(texts), T), PAD)
    for i, t in enumerate(texts):
        ids[i, :len(enc(t))] = torch.tensor(enc(t))
    return ids

class TinyPolicy(nn.Module):
    """A GRU language model over responses, conditioned on the prompt id through its initial hidden state."""
    def __init__(self, n_prompts, vocab_size, d=32):
        super().__init__()
        self.prompt = nn.Embedding(n_prompts, d)
        self.emb = nn.Embedding(vocab_size, d)
        self.gru = nn.GRU(d, d, batch_first=True)
        self.out = nn.Linear(d, vocab_size)

    def seq_logprob(self, prompt_ids, resp_ids):
        """Sum of log π(token | prompt, previous tokens) over the response (pads ignored)."""
        bos = torch.full((len(resp_ids), 1), PAD)                        # a start symbol
        inp = torch.cat([bos, resp_ids[:, :-1]], dim=1)
        h, _ = self.gru(self.emb(inp), self.prompt(prompt_ids)[None])
        logp = F.log_softmax(self.out(h), -1).gather(-1, resp_ids[..., None]).squeeze(-1)
        return (logp * (resp_ids != PAD)).sum(1)

prompts = torch.arange(P)
Yc, Yr = to_batch(helpful), to_batch(curt)
torch.manual_seed(0)
policy = TinyPolicy(P, len(vocab))
opt = torch.optim.Adam(policy.parameters(), lr=0.03)
Yo = to_batch(rambling)
for step in range(200):                                          # SFT on all three styles, equally weighted
    loss = -(policy.seq_logprob(prompts, Yc) + policy.seq_logprob(prompts, Yr) + policy.seq_logprob(prompts, Yo)).mean() / 3
    opt.zero_grad(); loss.backward(); opt.step()
reference = copy.deepcopy(policy).eval()
for p_ in reference.parameters():
    p_.requires_grad_(False)
with torch.no_grad():
    lc, lr_ = reference.seq_logprob(prompts, Yc), reference.seq_logprob(prompts, Yr)
    lo = reference.seq_logprob(prompts, Yo)
print("SFT/reference model, per prompt:")
for i in range(P):
    print(f"  prompt {i}: P(helpful) = {lc[i].exp():.3f}   P(curt) = {lr_[i].exp():.3f}   P(rambling) = {lo[i].exp():.3f}")

After SFT the model gives the three styles similar probability (about a third each — together almost all of the
probability mass). Now the DPO loss, exactly as written above:

In [ ]:
def dpo_loss(policy, reference, prompts, chosen, rejected, beta):
    pi_c, pi_r = policy.seq_logprob(prompts, chosen), policy.seq_logprob(prompts, rejected)
    with torch.no_grad():
        ref_c, ref_r = reference.seq_logprob(prompts, chosen), reference.seq_logprob(prompts, rejected)
    reward_c = beta * (pi_c - ref_c)                  # implicit rewards
    reward_r = beta * (pi_r - ref_r)
    loss = -F.logsigmoid(reward_c - reward_r).mean()
    stats = {"loss": loss.item(), "logp_chosen": pi_c.mean().item(), "logp_rejected": pi_r.mean().item(),
             "reward_chosen": reward_c.mean().item(), "reward_rejected": reward_r.mean().item(),
             "margin": (reward_c - reward_r).mean().item(), "accuracy": (reward_c > reward_r).float().mean().item()}
    return loss, stats

loss0, s0 = dpo_loss(copy.deepcopy(reference), reference, prompts, Yc, Yr, beta=0.1)
print(f"at step 0 the policy IS the reference: margin {s0['margin']:.3f}, loss {s0['loss']:.4f} = log 2 = {math.log(2):.4f}")
assert abs(s0["loss"] - math.log(2)) < 1e-6

At step 0 every implicit reward is zero, so the loss is exactly `−log σ(0) = log 2`. Train:

In [ ]:
def train_dpo(beta, steps=60, lr=0.01, seed=0):
    torch.manual_seed(seed)
    pol = copy.deepcopy(reference)
    for p_ in pol.parameters():
        p_.requires_grad_(True)
    opt_ = torch.optim.Adam(pol.parameters(), lr=lr)
    history = []
    for step in range(steps):
        loss, stats = dpo_loss(pol, reference, prompts, Yc, Yr, beta)
        opt_.zero_grad(); loss.backward(); opt_.step()
        history.append(stats)
    return pol.eval(), history

t0 = time.time()
dpo_policy, hist = train_dpo(beta=0.1)
print(f"60 DPO steps in {time.time() - t0:.1f}s")
for k in (0, 10, 30, 59):
    h = hist[k]
    print(f"step {k:3d}: loss {h['loss']:.3f}  margin {h['margin']:6.2f}  accuracy {h['accuracy']:.2f}  "
          f"log π(chosen) {h['logp_chosen']:6.2f}  log π(rejected) {h['logp_rejected']:7.2f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
steps = range(len(hist))
a1.plot(steps, [h["logp_chosen"] for h in hist], label="log π(chosen)", color="#2f9e44")
a1.plot(steps, [h["logp_rejected"] for h in hist], label="log π(rejected)", color="#e03131")
a1.axhline(lc.mean().item(), color="#2f9e44", ls=":", label="reference, chosen")
a1.axhline(lr_.mean().item(), color="#e03131", ls=":", label="reference, rejected")
a1.set_xlabel("DPO step"); a1.set_ylabel("sequence log-probability"); a1.set_title("What DPO does to the log-probs"); a1.legend(fontsize=8)
a2.plot(steps, [h["reward_chosen"] for h in hist], label="implicit reward, chosen", color="#2f9e44")
a2.plot(steps, [h["reward_rejected"] for h in hist], label="implicit reward, rejected", color="#e03131")
a2.plot(steps, [h["margin"] for h in hist], label="margin", color="#3b5bdb", lw=2)
a2.set_xlabel("DPO step"); a2.set_ylabel("β · log(π / π_ref)"); a2.set_title("Implicit rewards and margin (β = 0.1)"); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()
assert hist[-1]["margin"] > 0 and hist[-1]["accuracy"] == 1.0 and hist[-1]["logp_rejected"] < lr_.mean().item() - 3

**Read the charts.** The margin grows and every pair is ranked correctly. Look at *how*: the rejected answers'
log-probability falls a long way, while the chosen ones move much less — DPO optimises the **gap**, not the
chosen probability. In real LLM training it is common to see the chosen log-prob *fall too* (just more slowly than
the rejected one); people watch for this, and variants such as adding an SFT term on the chosen answers (as ORPO and
some DPO recipes do) exist partly because of it. The implicit reward is a *relative* score against the reference,
not a calibrated quality number.

Sampled behaviour, before and after:

In [ ]:
@torch.no_grad()
def sample(pol, prompt_id, n=200, max_len=8, seed=0):
    g = torch.Generator().manual_seed(seed)
    h = pol.prompt(torch.full((n,), prompt_id))[None]
    tok_ = torch.full((n, 1), PAD); out = []
    done = torch.zeros(n, dtype=torch.bool)
    for _ in range(max_len):
        o, h = pol.gru(pol.emb(tok_), h)
        tok_ = torch.multinomial(F.softmax(pol.out(o[:, -1]), -1), 1, generator=g)
        tok_[done] = PAD
        out.append(tok_); done |= tok_.squeeze(1) == EOS
    return [" ".join(vocab[t] for t in row if t not in (PAD, EOS)) for row in torch.cat(out, 1).tolist()]

def helpful_rate(pol):
    rates = []
    for i in range(P):
        s = sample(pol, i, seed=i)
        rates.append(np.mean([x == helpful[i] for x in s]))
    return float(np.mean(rates))

print(f"helpful share of sampled answers: reference {helpful_rate(reference):.0%} -> after DPO {helpful_rate(dpo_policy):.0%}")
print("three samples for prompt 0 after DPO:", sample(dpo_policy, 0, n=3, seed=5))
assert helpful_rate(dpo_policy) > 0.9

## 5. The effect of β

`β` is the strength of the leash to the reference. The loss `−log σ(β · Δ)` (Δ = the log-ratio margin) is close to
zero once `β · Δ` is a few units, so β decides how big a gap counts as "enough": a **small β** keeps pushing until
the log-ratios are huge (the policy travels far from the reference), a **large β** is satisfied with a small gap and
stays closer.

In [ ]:
betas = [0.01, 0.1, 0.5, 2.0]
t0 = time.time()
beta_runs = {b: train_dpo(beta=b) for b in betas}
print(f"{len(betas)} DPO runs in {time.time() - t0:.1f}s")
beta_stats = {}
for b, (pol, h) in beta_runs.items():
    beta_stats[b] = {"log_ratio_margin": h[-1]["margin"] / b, "logp_rejected": h[-1]["logp_rejected"],
                     "helpful": helpful_rate(pol), "accuracy": h[-1]["accuracy"]}
    s_ = beta_stats[b]
    print(f"β={b:<5}: log-ratio margin {s_['log_ratio_margin']:6.1f}   log π(rejected) {s_['logp_rejected']:7.1f}   "
          f"helpful share {s_['helpful']:.0%}   pair accuracy {s_['accuracy']:.2f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
for b, (pol, h) in beta_runs.items():
    a1.plot([x["margin"] / b for x in h], label=f"β = {b}")
a1.set_xlabel("DPO step"); a1.set_ylabel("log-ratio margin Δ"); a1.set_title("Smaller β → pushes the log-ratios further"); a1.legend(fontsize=8)
a2.semilogx(betas, [beta_stats[b]["logp_rejected"] for b in betas], "o-", color="#e03131")
a2.axhline(lr_.mean().item(), color="grey", ls=":", label="reference")
a2.set_xlabel("β (log)"); a2.set_ylabel("final log π(rejected)"); a2.set_title("How hard the rejected answers were pushed down")
a2.legend(); plt.tight_layout(); plt.show()
assert beta_stats[0.01]["log_ratio_margin"] > 5 * beta_stats[2.0]["log_ratio_margin"]

Every β learns the ranking on this easy toy, and with only one good answer per prompt they all end up preferring it.
What β changes is **how far** the policy travels to get there: at β = 0.01 the rejected answers are driven to
absurdly low log-probabilities, at β = 2 the run stops much closer to the reference. On a real LLM that distance is
the risk — far from the SFT reference, text can degenerate or over-fit the pairs — so β is the knob for "how much
change do I trust these pairs to justify". Common starting values are β ≈ 0.1 (0.01–0.5 in practice), watching
reward accuracy, margins and a held-out quality eval together. DPO also needs a *decent SFT model first*: the
reference is the anchor, and regularising toward a model that does not follow instructions is pointless (`tn37`).

## 6. The alternatives, one line each (hedged)

| Method | One line | Needs | Use when |
|---|---|---|---|
| **IPO** | replaces DPO's log-sigmoid with a squared loss toward a fixed margin, so it cannot over-fit deterministic preferences | pairs + reference | DPO over-fits (margins run away, quality drops) |
| **KTO** | prospect-theory-style loss on *unpaired* thumbs up / down | single answers labelled good/bad + reference | your data is thumbs, not pairs — usually true in production (`tn19`) |
| **ORPO** | one stage: SFT loss + an odds-ratio penalty on the rejected answer; no reference model | pairs | you want SFT and preference tuning in one run, with less memory |
| **SimPO** | reference-free: uses the length-normalised average log-prob as the reward, plus a target margin | pairs | simpler/cheaper DPO; length normalisation counters verbosity |
| **GRPO** | RL: sample a group of answers per prompt, score them, advantage = (r − group mean) / std; no critic network | a reward *function* (tests pass, answer checks) | verifiable tasks — maths, code; the recipe behind recent reasoning models (`tn09`) |
| **PPO (RLHF)** | on-line RL against a learned reward model with a KL penalty to the reference | reward model + value model | a live reward signal, on-line exploration |

Reported gains between these methods are often small and benchmark-dependent; data quality and a good SFT start
matter more than the choice of loss.

GRPO's advantage in one line of code — the group mean is the baseline, so no value network is needed:

In [ ]:
rewards = torch.tensor([[1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0]])       # 8 sampled answers to one maths prompt: pass/fail
advantage = (rewards - rewards.mean(1, keepdim=True)) / (rewards.std(1, keepdim=True) + 1e-6)
print("rewards   :", rewards.tolist()[0])
print("advantages:", [round(a, 2) for a in advantage.tolist()[0]])
assert abs(advantage.sum().item()) < 1e-5              # above-average answers pushed up, below-average pushed down

## Try it yourself

**1.** Replace DPO by plain **SFT on the chosen answers only** (maximise `log π(chosen)`) for 60 steps. Compare the
final `log π(rejected)` with DPO's. What does SFT *not* teach?

In [ ]:
# Solution — try it yourself first, then run this
torch.manual_seed(0)
sft_only = copy.deepcopy(reference)
for p_ in sft_only.parameters():
    p_.requires_grad_(True)
opt_ = torch.optim.Adam(sft_only.parameters(), lr=0.01)
for _ in range(60):
    loss = -sft_only.seq_logprob(prompts, Yc).mean()
    opt_.zero_grad(); loss.backward(); opt_.step()
with torch.no_grad():
    print(f"SFT on chosen : log π(chosen) {sft_only.seq_logprob(prompts, Yc).mean():6.2f}   log π(rejected) {sft_only.seq_logprob(prompts, Yr).mean():6.2f}")
print(f"DPO (β=0.1)   : log π(chosen) {hist[-1]['logp_chosen']:6.2f}   log π(rejected) {hist[-1]['logp_rejected']:6.2f}")
print("SFT only pushes the chosen answer up; the rejected one falls only as a side effect of probability mass moving.")
print("DPO gets an explicit signal about what to avoid.")

**2.** Flip 2 of the 6 preference labels (label noise) and retrain DPO. What happens to those two prompts?

In [ ]:
# Solution — try it yourself first, then run this
Yc_noisy = to_batch(curt[:2] + helpful[2:])          # prompts 0 and 1: the curt answer is labelled 'chosen'
Yr_noisy = to_batch(helpful[:2] + curt[2:])
torch.manual_seed(0)
noisy = copy.deepcopy(reference)
for p_ in noisy.parameters():
    p_.requires_grad_(True)
opt_ = torch.optim.Adam(noisy.parameters(), lr=0.01)
for _ in range(60):
    loss, _ = dpo_loss(noisy, reference, prompts, Yc_noisy, Yr_noisy, beta=0.1)
    opt_.zero_grad(); loss.backward(); opt_.step()
with torch.no_grad():
    p_help = noisy.seq_logprob(prompts, Yc).exp()
print("P(helpful answer) per prompt after training on flipped labels for prompts 0 and 1:")
print([round(v, 3) for v in p_help.tolist()])
print("DPO learns the curt answer for the mislabelled prompts - it trusts the pairs completely - and, because")
print("all prompts share one network, the wrong signal can leak into the others too.")

**3.** Compute the DPO loss by hand for one pair: β = 0.1, policy log-probs (chosen −4, rejected −9), reference
log-probs (chosen −5, rejected −6). Then check it with `F.logsigmoid`.

In [ ]:
# Solution — try it yourself first, then run this
beta, pc, pr, rc, rr = 0.1, -4.0, -9.0, -5.0, -6.0
margin = beta * ((pc - rc) - (pr - rr))                     # 0.1 * (1 - (-3)) = 0.4
by_hand = -math.log(1 / (1 + math.exp(-margin)))
print(f"margin {margin:.2f}, loss by hand {by_hand:.4f}, torch {-F.logsigmoid(torch.tensor(margin)).item():.4f}")

## Say it in an interview

- **DPO in 30 seconds:** "RLHF trains a reward model on preference pairs and then runs PPO against it with a KL
  penalty to the SFT model — four models, on-line generation, finicky. DPO shows the KL-regularised optimum has a
  closed form, so you can skip the reward model and optimise the policy directly on (prompt, chosen, rejected):
  loss = −log σ(β[(log π(c) − log π_ref(c)) − (log π(r) − log π_ref(r))]). Two models, an offline loss, about the
  cost of SFT."
- **β:** the leash to the frozen reference; small β = bigger drift, large β = stays near SFT; ~0.1 is a common start.
- **What it can teach:** "better vs worse" — tone, concision, refusals — things easy to judge, hard to write
  (`tn37`). It needs a good SFT model first and cannot exceed the information in the pairs (offline, no exploration).
- **Reward models:** Bradley–Terry `σ(r_c − r_r)`; they learn labeller biases (length!) and get hacked when optimised
  hard — hence KL penalties, length control, ensembles and refreshed reward models.
- **Pick the method by the data:** pairs → DPO/IPO/SimPO; thumbs → KTO; one-stage & no reference → ORPO; a
  programmatic checker → GRPO.
- **Traps:** "DPO increases the probability of the chosen answer" (it increases the *gap*; chosen can fall too);
  skipping SFT; flipped/noisy labels are learned faithfully; after any tuning, re-run safety and general evals — the
  alignment tax today mostly shows up as over-refusal (`tn38`).

## Next

- [07 · Distillation and catastrophic forgetting](07_distillation_and_catastrophic_forgetting.ipynb)
- [08 · OpenAI fine-tuning API](08_openai_fine_tuning_api.ipynb) — hosted SFT, plus the DPO data format
- [LLM-as-judge and its biases](../12_evaluation/02_llm_as_judge_and_its_biases.ipynb) — where preference labels often come from
- [A/B testing and statistics for evals](../12_evaluation/05_ab_testing_and_statistics_for_evals.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)